# Training: nested context-state encoder, centralized critic, and both actors (Blotto)
All components are trained jointly from scratch under centralized training with decentralized execution, alternating a representation step on Eq. (3) (`Train_Encoder`) with a game step on the critic and both actors (`Train_Critic`, `Train_Player1`, `Train_Player2`). The defender's state GRU is trained through `Train_Player1`.

During training, the defender's behavior policy is its actor mixed with 25% uniform exploration, plus ε-greedy exploration that decays from 1.0 to 0.2. The attacker plays a uniformly random admissible allocation with probability ε (decaying from 1.0 toward 0), and otherwise the most likely action of its network.

In [ ]:
%run Initialize.ipynb

## Settings

In [ ]:
SEED = 20
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

GAMMA = 0.95
CRITIC_LR = 3e-5
PLAYER1_LR = 1e-5
PLAYER2_LR = 1e-5
ENCODER_LR = 5e-5
CRITIC_BATCH = 128
PLAYER1_BATCH = 128
PLAYER2_BATCH = 128
CONTEXT_BATCH = 128
TAU = 1e-4

# Eq. (3) weights and the Soft-InfoNCE temperature
LAMBDA_R = 0.25
LAMBDA_P = 0.25
LAMBDA_THETA = 0.25
SI_TEMPERATURE = 0.5

a_eps, a_eps_decay, a_eps_end = 1.0, 0.99, 0.2
u_eps, u_eps_decay, u_eps_end = 1.0, 0.99, 0.0

num_episodes = 10000
Update_Every = 5

## Networks, optimizers, replay buffers

In [ ]:
# Representation: context encoder, its auxiliary heads, EMA targets, and the state GRU
Context_Model = TSTransformer().to(device)
trans_pred  = nn.Sequential(nn.Linear(Embedding_Size + n_a + N, 256), nn.ReLU(), nn.Linear(256, 256), nn.ReLU(),
                            nn.Linear(256, Embedding_Size)).to(device)
reward_head = nn.Sequential(nn.Linear(Embedding_Size, 128), nn.ReLU(), nn.Linear(128, 1)).to(device)
proj_online = nn.Sequential(nn.Linear(Embedding_Size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
pred_online = nn.Sequential(nn.Linear(128, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Linear(256, 128)).to(device)
target_model = copy.deepcopy(Context_Model)
proj_target  = copy.deepcopy(proj_online)
for p_ in list(target_model.parameters()) + list(proj_target.parameters()):
    p_.requires_grad_(False)
target_model.eval(); proj_target.eval()
encoder_optimizer = optim.AdamW(list(Context_Model.parameters()) + list(trans_pred.parameters()) +
                                list(reward_head.parameters()) + list(proj_online.parameters()) +
                                list(pred_online.parameters()), lr=ENCODER_LR, amsgrad=True)

# Game: defender (with its state GRU), centralized critic, attacker
Z_S_ENC = StateGRU().to(device)
Player1 = Player1Actor().to(device)
player1_optimizer = optim.AdamW(list(Player1.parameters()) + list(Z_S_ENC.parameters()), lr=PLAYER1_LR, amsgrad=True)

critic = Critic().to(device)
critic_target = Critic().to(device)
critic_target.load_state_dict(critic.state_dict())
critic_optimizer = optim.AdamW(critic.parameters(), lr=CRITIC_LR, amsgrad=True)

Player2 = Player2Actor().to(device)
player2_optimizer = optim.AdamW(Player2.parameters(), lr=PLAYER2_LR, amsgrad=True)

exp_memory = ReplayMemory(10000)
context_memory = ReplayMemory(500)
All_Loss = np.zeros([4, num_episodes])          # encoder, critic, defender, attacker
env_steps = 0

## Training

In [ ]:
T = lambda v, dtype=torch.float32: torch.tensor(v, dtype=dtype, device=device)

for i_episode in tqdm(range(num_episodes)):

    Theta, Type_Theta, Weights, Reachable, reachable_actions = Sample_Context()
    x = np.random.beta(2.0, 2.0, size=N)
    z_s = torch.zeros(Z_S_Size, dtype=torch.float32, device=device)
    Context = np.zeros([max_steps, N * 2], dtype=np.float32)
    m2 = np.full(N * 2, 0.5)
    theta_Tensor, type_theta_Tensor = T(Theta), T(Type_Theta)

    for t in range(max_steps):
        mask = Context_Mask(t)
        Context_Tensor = T(Context).view(1, max_steps, N * 2)
        x_Tensor, m2_Tensor = T(x), T(m2)
        Context_Model.eval()
        with torch.no_grad():
            Embeding = Context_Model(Context_Tensor, mask).squeeze(0)
            a_dist = F.softmax(Player1(z_s, Embeding), dim=-1).cpu().numpy()
            u_dist = Player2(x_Tensor, m2_Tensor, type_theta_Tensor).cpu().numpy()
        a_dist = 0.75 * a_dist / a_dist.sum() + 0.25 * np.full(n_a, 1 / n_a)
        a_dist = a_dist / a_dist.sum()

        if np.random.rand() < a_eps:
            a = int(random.choices(range(n_a), weights=np.ones([n_a]) / n_a)[0])
        else:
            a = int(np.random.choice(n_a, p=a_dist))
        if np.random.rand() < u_eps:
            u = int(np.random.choice(reachable_actions))
        else:
            u = int(np.argmax(u_dist))

        next_x = next_state(x, actions[a], actions[u], Theta)
        r = reward(next_x, Theta)
        next_m2 = m2_update(m2, x, actions[u], next_x)
        y = Observation(next_x)
        with torch.no_grad():
            next_z_s = Z_S_ENC(z_s, T(y), T(actions[a]))
        env_steps += 1

        Context[t, :] = Context_Row(actions[a], y)

        if t > 0:
            next_mask = Context_Mask(t + 1)
            Next_Context_Tensor = T(Context).view(1, max_steps, N * 2)
            a_Tensor, u_Tensor, Reward_Tensor = T(a, torch.long), T(u, torch.long), T(r)
            exp_memory.push([x_Tensor.cpu(), T(next_x).cpu(), a_Tensor.cpu(), u_Tensor.cpu(), Reward_Tensor.cpu(),
                             type_theta_Tensor.cpu(), theta_Tensor.cpu(), m2_Tensor.cpu(), T(next_m2).cpu(),
                             Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu()])
            context_memory.push([Context_Tensor.cpu(), mask.cpu(), Next_Context_Tensor.cpu(), next_mask.cpu(),
                                 a_Tensor.cpu(), T(y).cpu(), Reward_Tensor.cpu(), theta_Tensor.cpu()])

        if i_episode >= 20 and t % Update_Every == 0:
            All_Loss[0, i_episode] += Train_Encoder()          # representation step, Eq. (3)
            All_Loss[1, i_episode] += Train_Critic()           # game step
            All_Loss[2, i_episode] += Train_Player1()
            All_Loss[3, i_episode] += Train_Player2()

        x, z_s, m2 = next_x, next_z_s, next_m2

    a_eps = max(a_eps_end, a_eps * a_eps_decay)
    u_eps = max(u_eps_end, u_eps * u_eps_decay)

## Save

In [ ]:
print(f"environment steps used in training: {env_steps:,d}")
torch.save(Context_Model.state_dict(), 'encoder.pth')
torch.save(Z_S_ENC.state_dict(), 'z_s_enc.pth')
torch.save(critic.state_dict(), 'critic.pth')
torch.save(Player1.state_dict(), 'player1.pth')
torch.save(Player2.state_dict(), 'player2.pth')